![image](https://raw.githubusercontent.com/IBM/watson-machine-learning-samples/master/cloud/notebooks/headers/watsonx-Prompt_Lab-Notebook.png)
# Use AutoAI RAG with SQL knowledge base reference

#### Disclaimers

- Use only Projects and Spaces that are available in watsonx context.


## Notebook content

This notebook contains the steps and code to demonstrate the usage of IBM AutoAI RAG with SQL database as knowledge source. The AutoAI RAG experiment conducted in this notebook uses simple exemplary data about employees of an imaginary company.

Some familiarity with Python is helpful. This notebook uses Python 3.12.


## Learning goal

The learning goals of this notebook are:

- Create an AutoAI RAG job that will find the best SQL RAG agent pattern based on provided SQL knowledge base.


## Contents

This notebook contains the following parts:

1. [Set up the environment](#Set-up-the-environment)
2. [RAG Optimizer definition](#RAG-Optimizer-definition)
3. [RAG Experiment run](#RAG-Experiment-run)
4. [Comparison and testing of RAG Patterns](#Comparison-and-testing-of-RAG-Patterns)
5. [Historical runs](#Historical-runs)
6. [Cleanup](#Cleanup)
7. [Summary and next steps](#Summary-and-next-steps)

<a id="Set-up-the-environment"></a>
## Set up the environment

Before you use the sample code in this notebook, you must perform the following setup tasks:

-  Create a <a href="https://cloud.ibm.com/catalog/services/watsonxai-runtime" target="_blank" rel="noopener no referrer">watsonx.ai Runtime Service</a> instance (a free plan is offered and information about how to create the instance can be found <a href="https://dataplatform.cloud.ibm.com/docs/content/wsj/getting-started/wml-plans.html?context=wx&audience=wdp" target="_blank" rel="noopener no referrer">here</a>).
-  Provide a knowledge base instance - PostgreSQL/MySQL/DB2

### Install and import the required modules and dependencies

In [1]:
%pip install -U "ibm-watsonx-ai[rag]>=1.4.6" | tail -n 1

#### Define credentials

Authenticate the watsonx.ai Runtime service on IBM Cloud Pak for Data. You need to provide the **admin's** `username` and the platform `url`.

In [2]:
import os

try:
    URL = os.environ["URL"]
except KeyError:
    URL = input("Please enter CPD instance url (hit enter): ")

try:
    USERNAME = os.environ["USERNAME"]
except KeyError:
    USERNAME = input("Please enter your username (hit enter): ")

Use the **admin's** `api_key` to authenticate watsonx.ai Runtime services:

In [3]:
import getpass

from ibm_watsonx_ai import Credentials

credentials = Credentials(
    username=USERNAME,
    api_key=getpass.getpass("Enter your watsonx.ai API key and hit enter: "),
    url=URL,
    instance_id="openshift",
    version="5.4",
)

Alternatively you can use the **admin's** `password`:

In [4]:
import getpass

from ibm_watsonx_ai import Credentials

if "credentials" not in locals() or not credentials.api_key:
    credentials = Credentials(
        username=USERNAME,
        password=getpass.getpass("Enter your watsonx.ai password and hit enter: "),
        url=URL,
        instance_id="openshift",
        version="5.4",
    )

#### Create `APIClient` instance

In [5]:
from ibm_watsonx_ai import APIClient

client = APIClient(credentials)

### Working with spaces

First, you need to create a space for your work. If you do not have a space already created, you can use `{PLATFORM_URL}/ml-runtime/spaces?context=icp4data` to create one.

- Click **New Deployment Space**
- Create an empty space
- Go to the space `Settings` tab
- Copy `Space GUID` into your env file or else enter it in the window which will show up after running below cell

**Tip**: You can also use SDK to prepare the space for your work. Find more information in the [Space Management sample notebook](https://github.com/IBM/watson-machine-learning-samples/blob/master/cpd5.0/notebooks/python_sdk/instance-management/Space%20management.ipynb).

**Action**: Assign the space ID below

In [6]:
import os

try:
    space_id = os.environ["SPACE_ID"]
except KeyError:
    space_id = input("Please enter your space_id (hit enter): ")

To be able to interact with all resources available in watsonx.ai, you need to set the **space** which you will be using.

In [7]:
client.set.default_space(space_id)

'SUCCESS'

<a id="RAG-Optimizer-definition"></a>
## RAG Optimizer definition

### Defining a connection to knowledge base

Provide id of connection to your knowledge database or create a new one. You can add connection on watsonx platform or type your credentials after running the code below.

In [8]:
knowledge_base_connection_id = (
    input(
        "Provide connection asset ID in your space. Skip this, if you wish to type credentials by hand and hit enter: "
    )
    or None
)

if knowledge_base_connection_id is None:
    try:
        db_type_name = os.environ["DB_TYPE"].upper()
    except KeyError:
        db_type_name = input(
            "Please enter your db type from provided ones: P [Postgres], D [DB2], M [MySQL]"
        ).upper()

    if db_type_name == "P" or db_type_name == "POSTGRES":
        db_type = "postgresql"
    elif db_type_name == "D" or db_type_name == "DB2":
        db_type = "db2"
    elif db_type_name == "M" or db_type_name == "MYSQL":
        db_type = "mysql"
    else:
        raise ValueError(
            "Unavailable db_type_name. Choose one of P [Postgres], D [DB2], M [MySQL]."
        )

    try:
        hostname = os.environ["DATABASE_HOST"].upper()
    except KeyError:
        hostname = input("Please enter hostname or IP address of your database: ")
    try:
        port = os.environ["DATABASE_PORT"]
    except KeyError:
        port = input("Please enter your database port number and hit enter: ")
    try:
        database = os.environ["DATABASE"]
    except KeyError:
        database = input("Please enter your database name and hit enter: ")
    try:
        username = os.environ["DATABASE_USERNAME"]
    except KeyError:
        username = input("Please enter your username and hit enter: ")
    try:
        password = os.environ["DATABASE_PASSWORD"]
    except KeyError:
        password = getpass.getpass("Please enter your password and hit enter: ")
    try:
        ssl = os.environ["DATABASE_SSL_CERTIFICATE"]
    except KeyError:
        ssl = getpass.getpass("Please enter your ssl certificate and hit enter: ")

    # Create connection
    db_data_source_type_id = client.connections.get_datasource_type_uid_by_name(db_type)
    details = client.connections.create(
        {
            client.connections.ConfigurationMetaNames.NAME: "Knowledge database connection",
            client.connections.ConfigurationMetaNames.DESCRIPTION: "Connection created by the sample notebook",
            client.connections.ConfigurationMetaNames.DATASOURCE_TYPE: db_data_source_type_id,
            client.connections.ConfigurationMetaNames.PROPERTIES: {
                "host": hostname,
                "port": port,
                "username": username,
                "password": password,
                "database": database,
                "ssl": True,
                "ssl_certificate": ssl,
            },
        }
    )

    knowledge_base_connection_id = client.connections.get_id(details)

try:
    schema_name = os.environ["SCHEMA"]
except KeyError:
    schema_name = (
        input(
            "Please enter name of the schema you want to use (if you are using basic schema, just hit enter): "
        )
        or "public"
    )

Creating connections...
SUCCESS


Define a reference to knowledge base.

In [9]:
from ibm_watsonx_ai.helpers import DatabaseLocation, DataConnection
from ibm_watsonx_ai.utils.autoai.knowledge_base import DatabaseKnowledgeBase

knowledge_base_references = [
    DatabaseKnowledgeBase(
        name="Sample notebook knowledge database",
        description="Base used in exemplary notebook from watsonx_ai_samples",
        connection=DataConnection(
            connection_asset_id=knowledge_base_connection_id,
            location=DatabaseLocation(schema_name=schema_name),
        ),
    )
]

### Defining a connection to test data

Define benchmarking question about your knowledge base. Replace the questions below.

In [10]:
benchmarking_data_IBM_page_content = [
    {
        "question": "Who earns the highest salary in the Engineering department?",
        "correct_answer": "Jack Thompson earns the highest salary in the Engineering department.",
    },
    {
        "question": "List all employees hired before 2019.",
        "correct_answer": "The employees hired before 2019 are Jack Thompson, Emma Davis, Daniel Brown, and Henry Clark.",
    },
    {
        "question": "What’s the average salary per department?",
        "correct_answer": "Engineering: $101,000; Marketing: $71,000; Finance: $77,500; Human Resources: $85,000; Sales: $69,000.",
    },
]

The code in the next cell uploads testing data to the bucket as a `json` file.

In [11]:
import json

test_filename = "benchmarking_data_kb_sample.json"

if not os.path.isfile(test_filename):
    with open(test_filename, "w") as json_file:
        json.dump(benchmarking_data_IBM_page_content, json_file, indent=4)

test_asset_details = client.data_assets.create(
    name=test_filename, file_path=test_filename
)

test_asset_id = client.data_assets.get_id(test_asset_details)
test_asset_id

Creating data asset...
SUCCESS


'01a06cea-ecba-7233-b388-09afa8116bf4'

Define connection information to testing data.

In [12]:
test_data_references = [DataConnection(data_asset_id=test_asset_id)]

### RAG Optimizer configuration

Provide the input information for AutoAI RAG optimizer:
- `name` - experiment name
- `description` - experiment description
- `max_number_of_rag_patterns` - maximum number of RAG patterns to create
- `optimization_metrics` - target optimization metrics

In [13]:
from ibm_watsonx_ai.experiment import AutoAI
from ibm_watsonx_ai.foundation_models.schema import (
    AutoAIRAGGenerationConfig,
    AutoAIRAGModelConfig,
)

experiment = AutoAI(
    credentials=credentials,
    space_id=space_id,
)

foundation_model = AutoAIRAGModelConfig(
    model_id="ibm/granite-4-h-small",
)

generation_config = AutoAIRAGGenerationConfig(
    foundation_models=[foundation_model],
)

rag_optimizer = experiment.rag_optimizer(
    name="AutoAI RAG - sample notebook - knowledge base",
    description="Experiment run in sample notebook",
    generation=generation_config,
    max_number_of_rag_patterns=3,
    optimization_metrics=[AutoAI.RAGMetrics.ANSWER_CORRECTNESS],
)

Configuration parameters can be retrieved via `get_params()`.

In [14]:
rag_optimizer.get_params()

{'name': 'AutoAI RAG - sample notebook - knowledge base',
 'description': 'Experiment run in sample notebook',
 'max_number_of_rag_patterns': 3,
 'optimization_metrics': ['answer_correctness'],
 'generation': {'foundation_models': [{'model_id': 'ibm/granite-4-h-small'}]}}

<a id="RAG-Experiment-run"></a>
## RAG Experiment run

Call the `run()` method to trigger the AutoAI RAG experiment. You can either use interactive mode (synchronous job) or background mode (asynchronous job) by specifying `background_mode=True`.

In [15]:
run_details = rag_optimizer.run(
    knowledge_base_references=knowledge_base_references,
    test_data_references=test_data_references,
    background_mode=False,
)



##############################################

Running '5f45462d-e956-4e94-a0b7-6d6e9dffc8b2'

##############################################


pending.....
running..........
completed
Training of '5f45462d-e956-4e94-a0b7-6d6e9dffc8b2' finished successfully.


You can use the `get_run_status()` method to monitor AutoAI RAG jobs in background mode.

In [16]:
rag_optimizer.get_run_status()

'completed'

<a id="Comparison-and-testing-of-RAG-Patterns"></a>
## Comparison and testing of RAG Patterns

You can list the trained patterns and information on evaluation metrics in the form of a Pandas DataFrame by calling the `summary()` method. You can use the DataFrame to compare all discovered patterns and select the one you like for further testing.

In [17]:
summary = rag_optimizer.summary()
summary

,mean_answer_correctness,retrieval.method,generation.model_id,agent.type
Pattern_Name,,,,
Pattern1,0.3755,query_engine,ibm/granite-4-h-small,sequential


Additionally, you can pass the `scoring` parameter to the summary method, to filter RAG patterns starting with the best.

```python
summary = rag_optimizer.summary(scoring="faithfulness")
```

In [18]:
rag_optimizer.get_run_details()

{'entity': {'hardware_spec': {'id': 'a6c4923b-b8e4-444c-9f43-8a7ec3020110',
   'name': 'L'},
  'knowledge_base_references': [{'description': 'Base used in exemplary notebook from watsonx_ai_samples',
    'name': 'Sample notebook knowledge database',
    'reference': {'connection': {'id': '01a06cea-c7fc-7076-a516-af3fd4e039fe'},
     'location': {'schema_name': 'public'},
     'type': 'connection_asset'},
    'type': 'database'}],
  'parameters': {'constraints': {'generation': {'foundation_models': [{'model_id': 'ibm/granite-4-h-small'}]},
    'max_number_of_rag_patterns': 3},
   'optimization': {'metrics': ['answer_correctness']},
   'output_logs': True},
  'results': [{'context': {'iteration': 0,
     'max_combinations': 1,
     'rag_pattern': {'composition_steps': ['model_selection',
       'sql_execution',
       'optimization'],
      'duration_seconds': 48,
      'location': {'evaluation_results': 'default_autoai_rag_out/5f45462d-e956-4e94-a0b7-6d6e9dffc8b2/Pattern1/evaluation_res

### Get selected pattern

Get the RAGPattern object from the RAG Optimizer experiment. By default, the RAGPattern of the best pattern is returned.

In [19]:
best_pattern_name = summary.index.values[0]
print("Best pattern is:", best_pattern_name)

best_pattern = rag_optimizer.get_pattern()

Best pattern is: Pattern1


The pattern details can be retrieved by calling the `get_pattern_details` method:

```python
rag_optimizer.get_pattern_details(pattern_name='Pattern2')
```

Query the RAGPattern locally, to test it.

In [20]:
from ibm_watsonx_ai.deployments import RuntimeContext

runtime_context = RuntimeContext(api_client=client)
inference_service_function = best_pattern.inference_service(runtime_context)[0]

In [21]:
question = "Which employees are based in New York?"

context = RuntimeContext(
    api_client=client,
    request_payload_json={"messages": [{"role": "user", "content": question}]},
)

inference_service_function(context)

{'body': {'choices': [{'index': 0,
'message': {'role': 'assistant',
'content': 'Based on the provided SQL query and result, the employees based in New York are:

1. Alice Johnson - Software Engineer
2. Frank Miller - Data Scientist
3. Jack Thompson - Software Architect'},
'reference_documents': [{'metadata': {'document_id': 'scheme_functional_test_1'}}]}]}}

### Deploy RAGPattern

Deployment is done by storing the defined RAG function and then by creating a deployed asset.

In [22]:
deployment_details = best_pattern.inference_service.deploy(
    name="AutoAI RAG deployment - ibm_watsonx_ai documentation",
    space_id=space_id,
    deploy_params={"tags": ["wx-autoai-rag"]},
)



######################################################################################

Synchronous deployment creation for id: '01a06cef-8d16-7040-87d9-030407e6bfe3' started

######################################################################################


initializing
Note: online_url and serving_urls are deprecated and will be removed in a future release. Use inference instead.
.......
ready


-----------------------------------------------------------------------------------------------
Successfully finished deployment creation, deployment_id='01a06cef-b191-75de-8af5-37050930ecf5'
-----------------------------------------------------------------------------------------------




### Test the deployed function

RAG service is now deployed in our space. To test our solution we can run the cell below. Questions have to be provided in the payload. Their format is provided below.

In [23]:
deployment_id = client.deployments.get_id(deployment_details)

payload = {"messages": [{"role": "user", "content": question}]}
score_response = client.deployments.run_ai_service(deployment_id, payload)

In [24]:
print(score_response["choices"][0]["message"]["content"])

Based on the provided SQL query and result, the employees who are based in New York are:

1. **Alice Johnson** - Software Engineer in the Engineering department
2. **Frank Miller** - Data Scientist in the Engineering department
3. **Jack Thompson** - Software Architect in the Engineering department

In [25]:
score_response["choices"][0]["message"]["content"]

Based on the provided SQL query and result, the employees who are based in New York are:\n\n1. **Alice Johnson** - Software Engineer in the Engineering department\n2. **Frank Miller** - Data Scientist in the Engineering department\n3. **Jack Thompson** - Software Architect in the Engineering department

<a id="Historical-runs"></a>
## Historical runs

In this section you learn to work with historical RAG Optimizer jobs (runs).

To list historical runs use the `list()` method and provide the `'rag_optimizer'` filter.

In [26]:
experiment.runs(filter="rag_optimizer").list()

,timestamp,run_id,state,auto_pipeline_optimizer name
0,2026-09-04T14:57:21.256Z,5f45462d-e956-4e94-a0b7-6d6e9dffc8b2,completed,AutoAI RAG - sample notebook - knowledge base


In [27]:
run_id = run_details["metadata"]["id"]
run_id

'5f45462d-e956-4e94-a0b7-6d6e9dffc8b2'

### Get executed optimizer's configuration parameters

In [28]:
experiment.runs.get_rag_params(run_id=run_id)

{'name': 'AutoAI RAG - sample notebook - knowledge base',
 'description': 'Experiment run in sample notebook',
 'max_number_of_rag_patterns': 3,
 'generation': {'foundation_models': [{'model_id': 'ibm/granite-4-h-small'}]},
 'optimization_metrics': ['answer_correctness']}

### Get historical rag_optimizer instance and training details

In [29]:
historical_opt = experiment.runs.get_rag_optimizer(run_id)

### List trained patterns for selected optimizer

In [30]:
historical_opt.summary()

,mean_answer_correctness,retrieval.method,generation.model_id,agent.type
Pattern_Name,,,,
Pattern1,0.3755,query_engine,ibm/granite-4-h-small,sequential


<a id="Cleanup"></a>
## Cleanup

To delete the current experiment, use the `cancel_run` method.

**Warning:** Be careful: once you delete an experiment, you will no longer be able to refer to it.

In [31]:
rag_optimizer.cancel_run(hard_delete=True)

'SUCCESS'

To delete the deployment, use the `delete` method. 

**Warning:** Keeping the deployment active may lead to unnecessary consumption of Compute Unit Hours (CUHs).

In [32]:
client.deployments.delete(deployment_id)

'SUCCESS'

If you want to clean up all created assets:
- experiments
- trainings
- pipelines
- model definitions
- models
- functions
- deployments

please follow up this sample [notebook](https://github.com/IBM/watson-machine-learning-samples/blob/master/cloud/notebooks/python_sdk/instance-management/Machine%20Learning%20artifacts%20management.ipynb).

<a id="Summary-and-next-steps"></a>
## Summary and next steps

You successfully completed this notebook!

You learned how to use `ibm-watsonx-ai` to run AutoAI RAG experiments. 

Check out our _<a href="https://ibm.github.io/watsonx-ai-python-sdk/samples.html" target="_blank" rel="noopener no referrer">Online Documentation</a>_ for more samples, tutorials, documentation, how-tos, and blog posts. 

### Authors and Maintainers

**Paweł Kocur**, Software Engineer at IBM watsonx.ai

**Rafał Chrzanowski**, Software Engineer at IBM watsonx.ai

Copyright © 2025-2026 IBM. This notebook and its source code are released under the terms of the MIT License.